# 📊 Notebook 02 — Full Benchmark Suite
**Phi-3 Mini 3.8B** | Kaggle T4 GPU

Measures: throughput (tok/s), latency percentiles (P50/P95/P99), VRAM usage
across precisions: FP16, INT8, INT4

> Run all cells top-to-bottom. Results auto-saved to `results/` and plotted inline.

In [ ]:
# Install & setup (skip if already done in Notebook 01)
import os
if not os.path.exists('llm-inference-engine'):
    !git clone https://github.com/yourusername/llm-inference-engine
    %cd llm-inference-engine
    !pip install -q -e .
else:
    %cd llm-inference-engine
    !pip install -q transformers accelerate bitsandbytes optimum mlflow seaborn

In [ ]:
import torch, gc, json, time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from IPython.display import display

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
os.makedirs('results', exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from src.engine import load_model_and_tokenizer, generate
from src.benchmark import run_benchmark, get_gpu_stats
from src.utils import get_gpu_memory_usage, clear_gpu_cache

# ── Configuration ────────────────────────────────────────────
MODEL_NAME  = 'microsoft/Phi-3-mini-4k-instruct'
CACHE_DIR   = '/kaggle/working/models'
WARMUP      = 3
EVAL_RUNS   = 30      # increase to 50 for more stable estimates
MAX_NEW_TOK = 256
PRECISIONS  = ['fp16', 'int8', 'int4']
print('Config ready.')

## 1. FP16 Benchmark

In [ ]:
gc.collect(); torch.cuda.empty_cache()

model_fp16, tokenizer = load_model_and_tokenizer(
    MODEL_NAME, precision='fp16', cache_dir=CACHE_DIR
)

mem_fp16 = get_gpu_memory_usage()
print(f'FP16 VRAM allocated: {mem_fp16["allocated_gb"]:.2f} GB')

result_fp16 = run_benchmark(
    model_fp16, tokenizer,
    precision='fp16',
    warmup_runs=WARMUP, eval_runs=EVAL_RUNS,
    max_new_tokens=MAX_NEW_TOK,
)
result_fp16['vram_gb'] = mem_fp16['allocated_gb']
print(f"FP16 → P50: {result_fp16['latency']['p50_ms']}ms | "
      f"TPS: {result_fp16['throughput']['avg_tokens_per_second']}")

In [ ]:
# Free FP16 model before loading quantized
del model_fp16; gc.collect(); torch.cuda.empty_cache()
print('FP16 model freed.')

## 2. INT8 Benchmark

In [ ]:
model_int8, _ = load_model_and_tokenizer(
    MODEL_NAME, precision='int8', cache_dir=CACHE_DIR
)
mem_int8 = get_gpu_memory_usage()
print(f'INT8 VRAM allocated: {mem_int8["allocated_gb"]:.2f} GB')

result_int8 = run_benchmark(
    model_int8, tokenizer,
    precision='int8',
    warmup_runs=WARMUP, eval_runs=EVAL_RUNS,
    max_new_tokens=MAX_NEW_TOK,
)
result_int8['vram_gb'] = mem_int8['allocated_gb']
print(f"INT8 → P50: {result_int8['latency']['p50_ms']}ms | "
      f"TPS: {result_int8['throughput']['avg_tokens_per_second']}")

In [ ]:
del model_int8; gc.collect(); torch.cuda.empty_cache()
print('INT8 model freed.')

## 3. INT4 Benchmark

In [ ]:
model_int4, _ = load_model_and_tokenizer(
    MODEL_NAME, precision='int4', cache_dir=CACHE_DIR
)
mem_int4 = get_gpu_memory_usage()
print(f'INT4 VRAM allocated: {mem_int4["allocated_gb"]:.2f} GB')

result_int4 = run_benchmark(
    model_int4, tokenizer,
    precision='int4',
    warmup_runs=WARMUP, eval_runs=EVAL_RUNS,
    max_new_tokens=MAX_NEW_TOK,
)
result_int4['vram_gb'] = mem_int4['allocated_gb']
print(f"INT4 → P50: {result_int4['latency']['p50_ms']}ms | "
      f"TPS: {result_int4['throughput']['avg_tokens_per_second']}")

In [ ]:
del model_int4; gc.collect(); torch.cuda.empty_cache()
print('INT4 model freed.')

## 4. Results Summary & Visualizations

In [ ]:
all_results = [result_fp16, result_int8, result_int4]
labels = ['FP16', 'INT8', 'INT4']

# ── Summary Table ───────────────────────────────────────────
rows = []
for r, lbl in zip(all_results, labels):
    rows.append({
        'Precision': lbl,
        'VRAM (GB)': round(r['vram_gb'], 2),
        'P50 (ms)': r['latency']['p50_ms'],
        'P95 (ms)': r['latency']['p95_ms'],
        'P99 (ms)': r['latency']['p99_ms'],
        'Avg TPS': r['throughput']['avg_tokens_per_second'],
    })

df = pd.DataFrame(rows)
display(df.style.highlight_max(subset=['Avg TPS'], color='lightgreen')
              .highlight_min(subset=['VRAM (GB)', 'P50 (ms)'], color='lightblue')
              .format(precision=2))

In [ ]:
# ── Plot 1: Throughput ───────────────────────────────────────
tps_vals  = [r['throughput']['avg_tokens_per_second'] for r in all_results]
vram_vals = [r['vram_gb'] for r in all_results]
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Throughput bar
bars = axes[0].bar(labels, tps_vals, color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for b, v in zip(bars, tps_vals):
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 1,
                 f'{v:.1f}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Throughput (tokens/sec)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Tokens / second')
axes[0].spines[['top','right']].set_visible(False)

# VRAM bar
bars2 = axes[1].bar(labels, vram_vals, color=colors, width=0.5, edgecolor='white', linewidth=1.2)
for b, v in zip(bars2, vram_vals):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height() + 0.05,
                 f'{v:.1f} GB', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('VRAM Usage (GB)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('VRAM (GB)')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Phi-3 Mini 3.8B — Kaggle T4 GPU', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/throughput_vram.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: Latency Percentiles ──────────────────────────────
p50 = [r['latency']['p50_ms'] for r in all_results]
p95 = [r['latency']['p95_ms'] for r in all_results]
p99 = [r['latency']['p99_ms'] for r in all_results]
x = np.arange(len(labels))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, p50, w, label='P50', color='#4C72B0')
ax.bar(x,     p95, w, label='P95', color='#55A868')
ax.bar(x + w, p99, w, label='P99', color='#C44E52')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Latency (ms)')
ax.set_title('Latency Percentiles by Precision\nPhi-3 Mini 3.8B | 256 output tokens | T4 GPU',
             fontweight='bold')
ax.legend(fontsize=11)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/latency_percentiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save all results to JSON ─────────────────────────────────
from src.benchmark import save_results_json
path = save_results_json({'results': all_results, 'model': 'phi3-mini-3.8b', 'gpu': torch.cuda.get_device_name(0)})
print(f'Results saved → {path}')

## ✅ Benchmark Complete

Key findings:
- **INT4** gives the best throughput and lowest VRAM — ideal for memory-constrained GPUs
- **FP16** gives best output quality at the cost of 2.7× more VRAM vs INT4
- **INT8** is a good middle ground — ~15% throughput gain over FP16 with minor quality loss

Plots saved to `results/`. Proceed to **Notebook 03** for a qualitative comparison of outputs.